# EDA: scraped forum posts (#31)

Explores `inputs/data/datasets.csv` (the scraped posts) before the first full run, to decide:

1. how many Problems the first full run uses (all 687 parseable, or a random subset), and
2. whether the 384 unparseable rows are scored as a separate Human group, and by which Detectors.

Not run by the pipeline. Needs only pandas and matplotlib (both on Colab). Run it from anywhere inside the repo;
on Colab, upload `datasets.csv` and `problems.csv` next to the notebook.

Every section ends with a **Read:** line to fill in after running. The decisions go in the last section.

## 1. Load

In [ ]:
# %% 1. Load. Read the same way modules/problems.py does: the file has a BOM, keep every cell as text.
import ast, re, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 80)

def find(name):
    for d in [Path.cwd(), *Path.cwd().parents]:
        for p in (d / "inputs/data" / name, d / name):
            if p.exists():
                return p
    raise FileNotFoundError(f"{name} not found: run inside the repo, or upload it next to the notebook")

raw = pd.read_csv(find("datasets.csv"), dtype=str, keep_default_na=False, encoding="utf-8-sig")
problems = pd.read_csv(find("problems.csv"), dtype=str, keep_default_na=False)

d = raw.copy()
d["parses"] = d["code_parses"] == "True"
d["post_type"] = d["code_source"]          # GLOSSARY: Post type
d["code_lines"] = pd.to_numeric(d["code_lines"], errors="coerce")
d["post_score"] = pd.to_numeric(d["post_score"], errors="coerce")
d["created"] = pd.to_datetime(d["post_created_utc"], errors="coerce")
d["edited"] = pd.to_datetime(d["post_last_edited_utc"], errors="coerce")
d["year"] = d["created"].dt.year
d["code_chars"] = d["code"].str.len()
d["group"] = np.where(d["parses"], "parses", "unparseable")

# Self-check against the numbers in #29 / #31.
assert len(d) == 1071, len(d)
assert d["parses"].sum() == 687 and (~d["parses"]).sum() == 384, d["parses"].value_counts()
assert len(problems) == 687, len(problems)
assert d["sample_id"].is_unique
print(sys.version)
print(d.shape)
d.head(3)

In [ ]:
# %% 1b. Empty cells per column (empty string = missing in this file).
(raw == "").sum().to_frame("empty").assign(share=lambda t: (t["empty"] / len(raw)).round(3)).T

**Read:** which columns have gaps that matter (dates, score, question_text)?

## 2. Counts by site, Post type, parses

In [ ]:
# %% 2. Counts. Parse rate per cell shows which source the parse filter thins out.
counts = pd.crosstab([d["site"], d["post_type"]], d["group"], margins=True)
counts["parse_rate"] = (counts["parses"] / counts["All"]).round(3)
display(counts)

kept = d[d["parses"]]
print("Main set (parses) share by site x Post type:")
display(pd.crosstab(kept["site"], kept["post_type"], margins=True, normalize=False))

fig, ax = plt.subplots(figsize=(7, 3))
pd.crosstab([d["site"], d["post_type"]], d["group"]).plot.barh(stacked=True, ax=ax)
ax.set_xlabel("rows"); ax.set_title("Rows by site x Post type"); plt.tight_layout(); plt.show()

**Read:** is any site or Post type so small (or so dominant) in the main set that per-split Results
(Human split) would be noise?

## 3. Re-check `code_parses` and why code fails

`code_parses` came with the file. Re-parse here with `ast.parse` on this Python (Colab is 3.13), and bucket the
failures. The bucket rules are rough heuristics: read the examples. Python 3.14 accepts `except A, B:`
(PEP 758), so Python 2 code with `except E, e:` parses there and the counts depend on the version.

In [ ]:
# %% 3. Re-parse and bucket the failures.
def parse_error(src):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")   # invalid escape sequences etc.
            ast.parse(src)
        return None
    except SyntaxError as e:          # IndentationError and TabError are SyntaxError subclasses
        return f"{type(e).__name__}: {e.msg}"
    except (ValueError, RecursionError, MemoryError) as e:   # null bytes, deep nesting
        return f"{type(e).__name__}"

d["error"] = d["code"].map(parse_error)
d["parses_here"] = d["error"].isna()
dis = d[d["parses"] != d["parses_here"]]
print(f"Python {sys.version.split()[0]}: {len(dis)} rows disagree with code_parses")
display(dis[["sample_id", "site", "post_type", "parses", "error"]].head(20))

PY2_PRINT = re.compile(r"^\s*print\s+[^\s(=.,)\]]", re.M)       # print "x" / print x
PY2_EXCEPT = re.compile(r"^\s*except\s+[\w.]+\s*,\s*\w+\s*:", re.M)  # except E, e:
PROMPT = re.compile(r"^\s*(>>>|In \[\d+\]:|\$ )", re.M)            # pasted REPL/shell session
ELLIPSIS = re.compile(r"^\s*(\.\.\.|…)\s*$", re.M)                 # elided lines

def bucket(row):
    if row["parses_here"]:
        return "parses on this Python"
    c, e = row["code"], row["error"] or ""
    if PY2_PRINT.search(c) or PY2_EXCEPT.search(c) or "Missing parentheses in call to 'print'" in e:
        return "python2"
    if PROMPT.search(c):
        return "pasted REPL/shell"
    if e.startswith(("IndentationError", "TabError")):
        return "indentation"
    if ELLIPSIS.search(c):
        return "elided lines (...)"
    if "never closed" in e or "unexpected EOF" in e or "was not closed" in e:
        return "cut off / unclosed"
    return "other"

d["fail_bucket"] = d.apply(bucket, axis=1)
u = d[~d["parses"]]
display(pd.crosstab(u["fail_bucket"], u["site"], margins=True))
print("Most common error messages (unparseable rows):")
display(u["error"].str.replace(r"\(.*?\)|'.*?'", "…", regex=True).value_counts().head(15))

In [ ]:
# %% 3b. One example per bucket (first 12 lines).
for b, g in u.groupby("fail_bucket"):
    r = g.iloc[0]
    print(f"=== {b}  ({len(g)} rows)  {r['sample_id']}  {r['error']}")
    print("\n".join(r["code"].splitlines()[:12]), "\n")

**Read:** do the disagreements (if any) change the 687 / 384 split? How much of the unparseable group is
Python 2 (fixable in principle, old) vs pasted output or fragments (not really code)?

## 4. Code length

LLM Generators write complete solutions. If Human code is mostly short fragments (often the case for
`question` posts), length alone could separate the labels, and a Detector would look better than it is.

In [ ]:
# %% 4. Length by group.
print("code_lines by site x Post type x group:")
display(d.groupby(["group", "site", "post_type"])["code_lines"]
          .describe(percentiles=[.1, .25, .5, .75, .9]).round(1))

assert (d["code_lines"] == d["code"].str.count("\n") + 1).mean() > 0.9, "code_lines is not a line count?"
for n in (1, 2, 3, 5):
    print(f"main set with <= {n} lines: {(kept['code_lines'] <= n).sum()}")
print("longest main-set snippets (lines):", kept["code_lines"].nlargest(5).tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
bins = np.logspace(0, np.log10(d["code_lines"].max() + 1), 30)
for (pt, g) in kept.groupby("post_type"):
    axes[0].hist(g["code_lines"], bins=bins, alpha=.5, label=pt)
axes[0].set_xscale("log"); axes[0].set_title("Main set: lines by Post type"); axes[0].legend()
for (gr, g) in d.groupby("group"):
    axes[1].hist(g["code_lines"], bins=bins, alpha=.5, label=gr)
axes[1].set_xscale("log"); axes[1].set_title("Lines: parses vs unparseable"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# %% 4b. The shortest main-set snippets: are they usable Human Samples?
for _, r in kept.nsmallest(8, "code_lines").iterrows():
    print(f"--- {r['sample_id']} {r['site']}/{r['post_type']} ({r['code_lines']} lines)")
    print(r["code"][:300])

**Read:** median length per Post type; how many tiny snippets (<= 2 lines); should a minimum length be a
filter rule (that would be a change to `modules/problems.py`, a separate ticket)?

## 5. Post year and LLM contamination

ChatGPT came out on 2022-11-30. Code posted or edited after that may already be LLM-written, which would put
LLM code under the Human label.

In [ ]:
# %% 5. Year by site and group; posts created or edited after ChatGPT.
CHATGPT = pd.Timestamp("2022-11-30")
display(pd.crosstab(d["year"], [d["group"], d["site"]], margins=True))

d["after_chatgpt"] = np.select(
    [d["created"] >= CHATGPT, d["edited"] >= CHATGPT, d["created"].isna()],
    ["created after", "edited after", "no date"], "before")
display(pd.crosstab([d["group"], d["site"]], d["after_chatgpt"], margins=True))

fig, ax = plt.subplots(figsize=(9, 3))
pd.crosstab(d["year"].astype("Int64"), d["group"]).plot.bar(ax=ax)
ax.set_title("Posts per year (ChatGPT released 2022-11-30)"); plt.tight_layout(); plt.show()

**Read:** how many main-set Human Samples were created / edited after 2022-11-30? Drop them, keep them,
or report them as their own Human split? Does the unparseable group skew older (Python 2), so that
"syntax errors" and "age" are confounded?

## 6. Formatting signals that could leak the label

LLM output has LF line endings, spaces, and ASCII. The `atc` and `whodunit` docstrings both name CRLF line endings
in human code as a label leak in their papers' data. Any of these in Human code is a free signal for a Detector.

In [ ]:
# %% 6. Formatting flags by group x site x Post type (share of rows).
c = d["code"]
flags = pd.DataFrame({
    "crlf": c.str.contains("\r\n", regex=False),
    "tab_indent": c.str.contains(r"^\t", regex=True, flags=re.M),
    "non_ascii": c.map(lambda s: not s.isascii()),
    "trailing_ws": c.str.contains(r"[ \t]+$", regex=True, flags=re.M),
    "has_comment": c.str.contains(r"#", regex=False),
    "has_docstring": c.str.contains('"""|\'\'\'', regex=True),
    "has_def": c.str.contains(r"^\s*def ", regex=True, flags=re.M),
    "has_class": c.str.contains(r"^\s*class ", regex=True, flags=re.M),
    "has_main_guard": c.str.contains("__main__", regex=False),
})
display(pd.concat([d[["group", "site", "post_type"]], flags], axis=1)
          .groupby(["group", "site", "post_type"]).mean().round(3))

In [ ]:
# %% 6b. Most imported top-level modules in the main set.
IMPORT = re.compile(r"^\s*(?:from\s+([\w]+)|import\s+([\w]+))", re.M)
mods = kept["code"].map(lambda s: {a or b for a, b in IMPORT.findall(s)}).explode().dropna()
display(mods.value_counts().head(20).to_frame("posts"))
print("main set with no import at all:", (kept["code"].map(lambda s: not IMPORT.search(s))).sum())

**Read:** any flag far from 0 that LLM code will not have (CRLF especially)? It would be worth a Formatter-check
style note, or stripping CRLF in the Problems conversion (separate ticket).

## 7. Duplicates and independence

Samples that share a statement, code or author are not independent, and a duplicated statement gives the
Generators the same Prompt twice.

In [ ]:
# %% 7. Duplicates.
d["statement"] = d["question_title"] + "\n\n" + d["question_text"]   # as modules/problems.py builds it
k = d[d["parses"]]
print("main set: duplicate statements:", k["statement"].duplicated(keep=False).sum(),
      "| duplicate code:", k["code"].duplicated(keep=False).sum(),
      "| same (site, question_id) twice:", k.duplicated(["site", "question_id"], keep=False).sum())
print("main set: empty question_text (title-only statement):", (k["question_text"].str.strip() == "").sum())
print("main set: same title on different sites:",
      k.groupby("question_title")["site"].nunique().gt(1).sum())

dup_q = k[k.duplicated(["site", "question_id"], keep=False)].sort_values(["site", "question_id"])
display(dup_q[["sample_id", "site", "question_id", "post_type", "code_lines", "question_title"]].head(12))

In [ ]:
# %% 7b. Author concentration (one prolific author = one style counted many times).
a = k.loc[k["author"] != "", "author"].value_counts()
print(f"main set: {a.size} distinct authors for {len(k)} rows; top 10 hold {a.head(10).sum()} rows")
display(a.head(10).to_frame("rows"))

**Read:** do question+answer pairs from one post exist (same statement, two Human Samples)? Keep both, or
one per question? Does any author dominate?

## 8. Statement length, post score, tags, license

In [ ]:
# %% 8. Other columns.
k = d[d["parses"]].assign(statement_chars=lambda t: t["statement"].str.len())
display(k.groupby(["site", "post_type"])[["statement_chars", "post_score"]].describe().round(0).T)
tags = k["tags"].str.split(";").explode().str.strip()
display(tags[tags != ""].value_counts().head(20).to_frame("posts"))
display(d.groupby("group")["license"].value_counts().unstack(0).fillna(0).astype(int))

**Read:** very short statements (Prompts that under-specify the task)? Licenses all compatible with
redistributing the Samples?

## 9. Sample size

The number of Problems N drives (a) Generator API cost and (b) how precise each Result is.

**Cost.** Fill in the Generators and prices you plan for the full run (`config.toml` lists a placeholder one).
Tokens are estimated from text length (~4 chars per token). Output tokens use the Human code length times a
factor, since LLM answers come with prose and fuller code. Edit the factor if you have real Generate output.

In [ ]:
# %% 9. Cost per N. EDIT these inputs.
GENERATORS = {                # name: (USD per 1M input tokens, USD per 1M output tokens)
    "gen-a": (0.15, 0.60),
    "gen-b": (3.00, 15.00),
}
PROMPT_FILES = 2              # "direct" + "style"
TEMPLATE_TOKENS = 150         # Prompt Template text around the statement (check inputs/templates)
OUTPUT_FACTOR = 3.0           # LLM answer tokens / Human code tokens
LLM_JUDGE = (0.25, 2.00)      # judge price; set None to skip. Scores every Sample incl. Formatted copies
EDIT_COUNTS = 2               # Human-fixed Samples per Direct Sample (e.g. Edit counts 2 and 5)

k = d[d["parses"]]
in_tok = k["statement"].str.len().mean() / 4 + TEMPLATE_TOKENS
out_tok = k["code"].str.len().mean() / 4 * OUTPUT_FACTOR
print(f"mean tokens per call: in {in_tok:.0f}, out {out_tok:.0f}")

rows = []
for n in (100, 200, 300, 400, 500, 687):
    calls = n * PROMPT_FILES
    gen_usd = sum(calls * (in_tok * pi + out_tok * po) / 1e6 for pi, po in GENERATORS.values())
    llm = n * len(GENERATORS) * (PROMPT_FILES + EDIT_COUNTS)   # Direct, Style-prompted, Human-fixed
    samples = (n + llm) * 2                                    # every Sample also gets a Formatted copy
    judge_usd = 0 if LLM_JUDGE is None else samples * 3 * (out_tok / OUTPUT_FACTOR * LLM_JUDGE[0] + 50 * LLM_JUDGE[1]) / 1e6
    rows.append(dict(N=n, generate_calls=calls * len(GENERATORS), samples_scored=samples,
                     generate_usd=round(gen_usd, 2), judge_usd=round(judge_usd, 2)))
display(pd.DataFrame(rows).set_index("N"))

**Precision.** A miss rate per Generator per Category is a share of N Samples. Its 95% interval half-width
is about 1.96 * sqrt(p(1-p)/N). The Outdated test compares that against the paper's miss rate, so the
half-width must be smaller than the gaps you want to see. Human splits (one site, one Post type) have fewer
Samples: their FPR is noisier still.

In [ ]:
# %% 9b. 95% half-width (percentage points) of a rate p measured on N Samples.
ps = (0.02, 0.05, 0.10, 0.25, 0.50)
ns = (50, 100, 200, 300, 400, 500, 687)
hw = pd.DataFrame({f"p={p}": [100 * 1.96 * np.sqrt(p * (1 - p) / n) for n in ns] for p in ps}, index=ns).round(1)
hw.index.name = "N"
display(hw)
print("Smallest Human splits in the main set (their FPR is measured on this many):")
display(pd.crosstab(k["site"], k["post_type"]))

**Subset rule, if N < 687.** Proposed: stratified random sample by site x Post type, proportional to the
counts above, fixed seed, so every Human split keeps its share. Implementing it is a change to
`modules/problems.py` (separate ticket).

In [ ]:
# %% 9c. Preview a stratified subset (does not write anything).
N, SEED = 300, 0
k = d[d["parses"]]
sub = k.groupby(["site", "post_type"]).sample(frac=N / len(k), random_state=SEED)   # rounds per stratum
print(len(sub), "Problems")
display(pd.concat({"all": k.groupby(["site", "post_type"]).size(),
                   "subset": sub.groupby(["site", "post_type"]).size()}, axis=1))
display(pd.concat({"all": k["code_lines"].describe(), "subset": sub["code_lines"].describe()}, axis=1).round(1))

**Read:** cost and half-width at each N; which N gives a half-width you can live with at a cost you can pay?

## 10. Unparseable rows as a separate Human group

The group needs **no generation**: it is Human Samples only. The measure is the FPR on unparseable Human code
next to the FPR on parseable Human code, per Detector. The gap shows how much syntax errors alone move a
Detector. Cost is scoring only (384 Samples, plus `llm_judge` API calls).

Which Detectors can read it (from the Detector docstrings):

| Detector | Input | Reads unparseable code? |
|---|---|---|
| `random_baseline` | none | yes (control) |
| `xgb_tfidf` | tiktoken token ids | yes |
| `droiddetect` | ModernBERT tokens | yes |
| `atc` | CodeLlama logits | yes |
| `llm_judge` | raw code in a Prompt | yes |
| `whodunit` | AST / radon stylometry | **no** (its docstring scores code it cannot read as 0.5) |

The group gets no Formatted copies (black needs parseable code) and no Human-fixed Samples (those come from
Direct Samples). The confound to check below: if unparseable code is older, shorter, or from other sites than
parseable code, an FPR gap is not "syntax errors alone".

In [ ]:
# %% 10. Parseable vs unparseable side by side.
cmp = pd.DataFrame({
    g: {"rows": len(x),
        "median lines": x["code_lines"].median(),
        "median year": x["year"].median(),
        "created after ChatGPT": (x["created"] >= CHATGPT).mean().round(3),
        "share answer": (x["post_type"] == "answer").mean().round(3),
        **{f"share {s}": (x["site"] == s).mean().round(3) for s in sorted(d["site"].unique())},
        "share CRLF": x["code"].str.contains("\r\n", regex=False).mean().round(3)}
    for g, x in d.groupby("group")})
display(cmp)
display(pd.crosstab(d["fail_bucket"], d["post_type"]).drop(index="parses on this Python", errors="ignore"))

**Read:** is the group comparable to the main set apart from parsing? If not, match it (e.g. compare against
parseable rows of the same site / Post type / era) before reading the FPR gap as a syntax effect.

## 11. Decisions (fill in after running)

- **Sample size N:** TBD
- **Subset rule:** TBD (all 687 / stratified by site x Post type, seed …)
- **Posts after 2022-11-30:** TBD (keep / drop / separate Human split)
- **Unparseable group:** TBD (score separately with `random_baseline`, `xgb_tfidf`, `droiddetect`, `atc`, `llm_judge`; not `whodunit` / skip)
- **Follow-up tickets:** TBD (e.g. subset rule in `modules/problems.py`, CRLF, min length, duplicates)